In [1]:
from vllm import LLM, SamplingParams
from pprint import pprint
from cs336_alignment.my_sft_toolfunc import *
from cs336_alignment.drgrpo_grader import r1_zero_reward_fn
import torch

INFO 12-10 02:36:54 __init__.py:190] Automatically detected platform cuda.


/home/nova/cs336/assignment5-alignment/cs336_alignment/drgrpo_grader.py:45: SyntaxWarning: invalid escape sequence '\{'
  m = re.search("^\\\\text\{(?P<text>.+?)\}$", answer)
/home/nova/cs336/assignment5-alignment/cs336_alignment/drgrpo_grader.py:320: SyntaxWarning: invalid escape sequence '\%'
  string = string.replace("\%", "")
/home/nova/cs336/assignment5-alignment/cs336_alignment/drgrpo_grader.py:673: SyntaxWarning: invalid escape sequence '\^'
  BAD_REGEXES = ["\^[0-9]+\^", "\^[0-9][0-9]+"]
/home/nova/cs336/assignment5-alignment/cs336_alignment/drgrpo_grader.py:673: SyntaxWarning: invalid escape sequence '\^'
  BAD_REGEXES = ["\^[0-9]+\^", "\^[0-9][0-9]+"]
/home/nova/cs336/assignment5-alignment/cs336_alignment/drgrpo_grader.py:753: SyntaxWarning: invalid escape sequence '\d'
  p1 = re.compile("(\d)(,)(\d\d\d)($|\D)")
/home/nova/cs336/assignment5-alignment/cs336_alignment/drgrpo_grader.py:768: SyntaxWarning: invalid escape sequence '\{'
  m = re.search("^\\\\text\{(?P<text>.+?)\}$"

In [2]:
qwen_math = LLM(model="/home/nova/cs336/assignment5-alignment/models/Qwen2.5-Math-1.5B")

INFO 12-10 02:37:15 config.py:542] This model supports multiple tasks: {'reward', 'classify', 'embed', 'generate', 'score'}. Defaulting to 'generate'.
INFO 12-10 02:37:15 llm_engine.py:234] Initializing a V0 LLM engine (v0.7.2) with config: model='/home/nova/cs336/assignment5-alignment/models/Qwen2.5-Math-1.5B', speculative_config=None, tokenizer='/home/nova/cs336/assignment5-alignment/models/Qwen2.5-Math-1.5B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execu

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 12-10 02:37:32 model_runner.py:1115] Loading model weights took 2.8797 GB
INFO 12-10 02:37:34 worker.py:267] Memory profiling takes 1.93 seconds
INFO 12-10 02:37:34 worker.py:267] the current vLLM instance can use total_gpu_memory (8.00GiB) x gpu_memory_utilization (0.90) = 7.20GiB
INFO 12-10 02:37:34 worker.py:267] model weights take 2.88GiB; non_torch_memory takes 0.03GiB; PyTorch activation peak memory takes 1.40GiB; the rest of the memory reserved for KV Cache is 2.89GiB.
INFO 12-10 02:37:35 executor_base.py:110] # CUDA blocks: 6768, # CPU blocks: 9362
INFO 12-10 02:37:35 executor_base.py:115] Maximum concurrency for 4096 tokens per request: 26.44x
INFO 12-10 02:37:36 model_runner.py:1434] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_utiliz

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:21<00:00,  1.63it/s]

INFO 12-10 02:37:57 model_runner.py:1562] Graph capturing finished in 21 secs, took 0.14 GiB
INFO 12-10 02:37:57 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 25.01 seconds


In [3]:
sampling_params = SamplingParams(
    temperature=1.0,
    top_p=1.0, 
    max_tokens=1024, 
    stop=["</answer>"],
    include_stop_str_in_output = True
)

In [4]:
qaiter = read_jsonl("/home/nova/cs336/assignment5-alignment/data/gsm8k/test.jsonl")
limited_qaiter = get_limited_iter(qaiter, 100)  # 把读文件限制设置在这里，防止一下读太多，后续操作均在可以设置这个限制的前提下进行
qas = get_qa_list(limited_qaiter)

In [5]:
for qa in qas:
    a = qa["answer"]
    true_answer = a.split("#### ",1)[1] if "#### " in a else ""
    qa["ground_truth"] = true_answer

In [6]:
prompts = []
with open("/home/nova/cs336/assignment5-alignment/cs336_alignment/prompts/r1_zero.prompt", encoding="utf-8") as f:
    template = f.read()

for qa in qas:
    q = qa["question"]
    prompts.append(template.format(question=q))

In [7]:
request_outputs = qwen_math.generate(prompts,sampling_params=sampling_params)

Processed prompts: 100%|██████████| 100/100 [00:45<00:00,  2.22it/s, est. speed input: 336.54 toks/s, output: 463.26 toks/s]


In [ ]:
assert len(request_outputs) == len(qas) , "模型生成的回答数与投入的数据量不符？？"

In [ ]:
for i in range(len(qas)):
    r = request_outputs[i]
    qa = qas[i]
    assert r.prompt == template.format(question=qa["question"]), "输入的问题与输出中记录的不同？？"
    #print(generated_answer)
    
    generated_answer = r.outputs[0].text
    qas[i]["generated_answer"] = generated_answer

In [ ]:
output_path = "output.jsonl"

with open(output_path, "w", encoding="utf-8") as f:
    for record in qas:
        f.write(json.dumps(record, ensure_ascii=False))
        f.write("\n")

In [ ]:
i = 43
#print(qas[i])

for i in range(100):
    print(r1_zero_reward_fn(qas[i]["generated_answer"],qas[i]["ground_truth"]))
    print(template.format(question=qas[i]["question"]))
    print(qas[i]["generated_answer"])
    print("--------------------------------------------------")

In [ ]:
for qa in qas:
    reward = r1_zero_reward_fn(qa["generated_answer"],qa["answer"])
    qa.update(reward)

In [ ]:
qas[1]

In [ ]:
from typing import Callable
def evaluate_vllm(
    vllm_model: LLM,
    reward_fn: Callable[[str, str], dict[str, float]],
    prompts: list[str],
    eval_sampling_params: SamplingParams
) -> None:
    """
    对于已知的vllm大模型和采样参数，以及一组给定的输入，根据给定的奖励（打分）函数，评估模型回答的情况，并将结果序列化后存盘。
    """


In [ ]:
import transformers
print(transformers.__version__)